# Amazon Reviews Sentiment Analysis — Deep Learning

**Goal:** Build a deep learning NLP model to classify Amazon customer reviews as positive or negative.

### Project requirements implemented
- NLTK preprocessing: stopword removal, punctuation removal, and `WordNetLemmatizer`.
- Train-only `Tokenizer.fit_on_texts()` to prevent data leakage.
- `pad_sequences` for sequence preparation.
- TensorFlow/Keras `Sequential` model with `Embedding` → `Bidirectional(LSTM)` → `Dropout` → `Dense(sigmoid)`.
- Training/validation Accuracy and Loss plots.
- Confusion Matrix and Classification Report.
- A local simulation of Aspect-Based Sentiment Analysis / entity extraction to represent the Google Cloud NLP integration concept without requiring credentials.

> **Important:** The Google Cloud section is a simulation. It does not claim to call the Google Cloud Natural Language API unless valid credentials are configured.


In [ ]:
# ============================================================
# 1. Imports
# ============================================================
import os
import re
import string
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
import tensorflow as tf

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


In [ ]:
# ============================================================
# 2. Download NLTK resources
# ============================================================
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")


## 3. Load the Amazon Reviews dataset

This notebook is designed to work with the common Amazon Reviews CSV format containing:
- `reviewText`: customer review text
- `overall`: star rating from 1 to 5

If your Kaggle dataset uses a different filename, update `DATA_PATH` below.


In [ ]:
# ============================================================
# 3. Locate and load CSV data on Kaggle
# ============================================================
import glob

csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)

print("CSV files found:")
for f in csv_files[:20]:
    print(f)

# Try to automatically find an Amazon review CSV.
amazon_candidates = [
    f for f in csv_files
    if "review" in os.path.basename(f).lower()
    or "amazon" in f.lower()
]

if not amazon_candidates:
    raise FileNotFoundError(
        "No suitable CSV was found under /kaggle/input/. "
        "Attach/select the Amazon Reviews dataset in the Kaggle notebook."
    )

DATA_PATH = amazon_candidates[0]
print("\nUsing:", DATA_PATH)

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())


## 4. Select the text and rating columns

The sentiment target is created from the star rating:
- **1–2 stars → Negative (0)**
- **4–5 stars → Positive (1)**
- **3 stars → removed as neutral**

This makes the task a binary sentiment classification problem.


In [ ]:
# ============================================================
# 4. Prepare text and sentiment labels
# ============================================================
TEXT_COL = "reviewText"
RATING_COL = "overall"

if TEXT_COL not in df.columns or RATING_COL not in df.columns:
    raise ValueError(
        f"Expected columns '{TEXT_COL}' and '{RATING_COL}'. "
        f"Available columns: {df.columns.tolist()}"
    )

data = df[[TEXT_COL, RATING_COL]].copy()
data = data.dropna(subset=[TEXT_COL, RATING_COL])

data[RATING_COL] = pd.to_numeric(data[RATING_COL], errors="coerce")
data = data.dropna(subset=[RATING_COL])

# Remove neutral 3-star reviews
data = data[data[RATING_COL] != 3].copy()

data["sentiment"] = (data[RATING_COL] >= 4).astype(int)

# Remove duplicate review texts
data = data.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)

print("Final shape:", data.shape)
print("\nSentiment distribution:")
print(data["sentiment"].value_counts())
display(data.head())


In [ ]:
# ============================================================
# 5. Basic EDA
# ============================================================
plt.figure(figsize=(6, 4))
sns.countplot(data=data, x="sentiment")
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment (0 = Negative, 1 = Positive)")
plt.ylabel("Number of Reviews")
plt.show()

print(data["sentiment"].value_counts(normalize=True).rename("percentage"))


## 6. Train / Validation / Test split

**The split happens before tokenizer fitting.**

This is important for avoiding data leakage: the tokenizer vocabulary will be learned from the training reviews only. Validation and test text will only be transformed using that already-fitted tokenizer.


In [ ]:
# ============================================================
# 6. Split BEFORE tokenization
# ============================================================
X = data[TEXT_COL].astype(str)
y = data["sentiment"].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))


## 7. NLTK Text Preprocessing

The cleaning pipeline uses NLTK for:
1. Lowercasing
2. Removing punctuation
3. Removing stopwords
4. Lemmatization with `WordNetLemmatizer`

The preprocessing function is applied consistently to train, validation, and test text.


In [ ]:
# ============================================================
# 7. NLTK preprocessing
# ============================================================
STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

def preprocess_text(text):
    text = str(text).lower()

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Keep alphabetic tokens and remove stopwords
    tokens = [
        word for word in text.split()
        if word.isalpha() and word not in STOP_WORDS
    ]

    # Lemmatization
    tokens = [
        LEMMATIZER.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)

X_train_clean = X_train.apply(preprocess_text)
X_val_clean = X_val.apply(preprocess_text)
X_test_clean = X_test.apply(preprocess_text)

display(pd.DataFrame({
    "original": X_train.head(5).values,
    "cleaned": X_train_clean.head(5).values
}))


## 8. Tokenization and Padding — without Data Leakage

The tokenizer is **fitted only on `X_train_clean`**.

```python
tokenizer.fit_on_texts(X_train_clean)
```

Validation and test data are transformed with `texts_to_sequences()` only.

This prevents words from the validation/test sets from influencing the learned vocabulary.


In [ ]:
# ============================================================
# 8. Tokenization — FIT ON TRAIN ONLY
# ============================================================
MAX_WORDS = 20000
MAX_LEN = 200

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

# IMPORTANT: fit only on training text
tokenizer.fit_on_texts(X_train_clean)

X_train_seq = tokenizer.texts_to_sequences(X_train_clean)
X_val_seq = tokenizer.texts_to_sequences(X_val_clean)
X_test_seq = tokenizer.texts_to_sequences(X_test_clean)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

print("Original vocabulary size:", len(tokenizer.word_index) + 1)
print("Vocabulary used by model:", vocab_size)
print("Train padded shape:", X_train_pad.shape)
print("Validation padded shape:", X_val_pad.shape)
print("Test padded shape:", X_test_pad.shape)


## 9. Deep Learning Model

Required architecture:

**Embedding → Bidirectional LSTM → Dropout → Dense(sigmoid)**

- `Embedding` learns a dense representation of words.
- `Bidirectional(LSTM)` reads the sequence in both directions.
- `Dropout` reduces overfitting.
- `Dense(1, activation="sigmoid")` produces a probability for the positive class.


In [ ]:
# ============================================================
# 9. Build TensorFlow / Keras Deep Learning model
# ============================================================
EMBEDDING_DIM = 128

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_LEN
    ),
    Bidirectional(
        LSTM(64, return_sequences=False)
    ),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
# ============================================================
# 10. Train the model
# ============================================================
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

EPOCHS = 8
BATCH_SIZE = 64

history = model.fit(
    X_train_pad,
    y_train.values,
    validation_data=(X_val_pad, y_val.values),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1
)


## 11. Training vs Validation — Accuracy and Loss


In [ ]:
# ============================================================
# 11. Training curves
# ============================================================
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(8, 5))
plt.plot(history_df["accuracy"], label="Training Accuracy")
plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="Training Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


## 12. Test Set Evaluation

The test set is kept separate from training and validation.

We report:
- Accuracy
- Classification Report
- Confusion Matrix


In [ ]:
# ============================================================
# 12. Evaluate on test set
# ============================================================
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test.values,
    verbose=0
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


In [ ]:
# ============================================================
# 13. Predictions + Classification Report
# ============================================================
y_prob = model.predict(X_test_pad, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Negative", "Positive"],
        digits=4
    )
)


In [ ]:
# ============================================================
# 14. Confusion Matrix
# ============================================================
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## 15. Real-Time Sentiment Prediction

The trained pipeline can now process a new review:
**raw text → NLTK preprocessing → tokenizer → padding → model → sentiment probability**


In [ ]:
# ============================================================
# 15. Real-time style prediction function
# ============================================================
def predict_sentiment(review_text):
    cleaned = preprocess_text(review_text)
    sequence = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(
        sequence,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

    probability = float(model.predict(padded, verbose=0)[0][0])
    sentiment = "Positive" if probability >= 0.5 else "Negative"

    return {
        "review": review_text,
        "cleaned_text": cleaned,
        "sentiment": sentiment,
        "positive_probability": round(probability, 4)
    }

examples = [
    "This product is excellent and works perfectly!",
    "The product stopped working after a few days and I am very disappointed."
]

for review in examples:
    print(predict_sentiment(review))


## 16. Google Cloud NLP API — Local Simulation

The project description mentions Google Cloud NLP API integration and Aspect-Based Sentiment Analysis / Entity Extraction.

To keep this notebook runnable on Kaggle without exposing Google Cloud credentials, the following cell provides a **local simulation**.

It extracts simple product-related aspects using keyword matching and assigns each aspect a sentiment based on the sentiment model.

> This is a simulation of the integration concept, **not an actual Google Cloud API call**.


In [ ]:
# ============================================================
# 16. Google Cloud NLP / Aspect-Based Sentiment Simulation
# ============================================================
ASPECT_KEYWORDS = {
    "battery": ["battery", "charge", "charging", "power"],
    "quality": ["quality", "material", "build"],
    "price": ["price", "cost", "expensive", "cheap", "value"],
    "delivery": ["delivery", "shipping", "arrived", "package"],
    "performance": ["performance", "speed", "fast", "slow", "works"],
    "design": ["design", "look", "size", "color"]
}

def extract_aspects(text):
    cleaned = preprocess_text(text)
    found_aspects = []

    for aspect, keywords in ASPECT_KEYWORDS.items():
        if any(keyword in cleaned.split() for keyword in keywords):
            found_aspects.append(aspect)

    return found_aspects

def simulate_google_cloud_nlp(text):
    sentiment_result = predict_sentiment(text)
    aspects = extract_aspects(text)

    aspect_results = [
        {
            "aspect": aspect,
            "sentiment": sentiment_result["sentiment"],
            "positive_probability": sentiment_result["positive_probability"]
        }
        for aspect in aspects
    ]

    return {
        "text": text,
        "document_sentiment": sentiment_result["sentiment"],
        "document_positive_probability": sentiment_result["positive_probability"],
        "entities_or_aspects": aspect_results
    }

sample_review = (
    "The battery lasts all day and the performance is excellent, "
    "but the price is expensive."
)

simulate_google_cloud_nlp(sample_review)


### Optional: Real Google Cloud Natural Language API integration

If you later add valid Google Cloud credentials and enable the Natural Language API, this project can be extended with the official client library.

Do **not** put service-account keys directly inside a Kaggle notebook or GitHub repository.


In [ ]:
# ============================================================
# 17. Optional Google Cloud API example (NOT executed)
# ============================================================
# pip install google-cloud-language

# from google.cloud import language_v1
#
# def google_cloud_sentiment(text):
#     client = language_v1.LanguageServiceClient()
#
#     document = language_v1.Document(
#         content=text,
#         type_=language_v1.Document.Type.PLAIN_TEXT,
#         language="en"
#     )
#
#     response = client.analyze_sentiment(
#         request={"document": document}
#     )
#
#     return {
#         "score": response.document_sentiment.score,
#         "magnitude": response.document_sentiment.magnitude
#     }
#
# NOTE:
# Authentication must be configured securely through Google Cloud.


## 18. Save the trained model

The model and tokenizer can be saved for later inference or deployment.


In [ ]:
# ============================================================
# 18. Save model + tokenizer
# ============================================================
import pickle

MODEL_PATH = "/kaggle/working/amazon_sentiment_bilstm.keras"
TOKENIZER_PATH = "/kaggle/working/amazon_sentiment_tokenizer.pkl"

model.save(MODEL_PATH)

with open(TOKENIZER_PATH, "wb") as f:
    pickle.dump(tokenizer, f)

print("Saved:")
print(MODEL_PATH)
print(TOKENIZER_PATH)


# Conclusion

This notebook implements an end-to-end Amazon review sentiment pipeline:

**Customer Review → NLTK Cleaning → Train-only Tokenization → Padding → Embedding → Bidirectional LSTM → Dropout → Sigmoid → Sentiment**

The workflow also includes:
- leakage-safe preprocessing
- training/validation curves
- test evaluation
- confusion matrix
- classification report
- real-time-style prediction
- a local simulation of aspect/entity sentiment analysis
- optional Google Cloud NLP integration
